# Statistical detection methods

This code demonstrates a deep learning-based approach to detecting adversarial examples using two detection mechanisms: Statistical Anomaly Detection and Neural Fingerprinting.

This code sets up the environment and prepares the MNIST dataset for use in a statistical detection experiment. It begins by importing the necessary libraries, including TensorFlow and Keras for model building, NumPy for numerical operations, and scikit-learn for evaluation metrics. The MNIST dataset, which contains 70,000 grayscale images of handwritten digits (0–9), is then loaded and split into a training set of 60,000 and a test set of 10,000. To standardise the inputs, the pixel values are converted from integers (0–255) to floating-point numbers between 0 and 1, which improves stability during training. A channel dimension is added so that the images, originally shaped (num_samples, 28, 28), become (num_samples, 28, 28, 1), matching the expected input format for convolutional networks. Finally, the digit labels are one-hot encoded, transforming each integer label into a 10-element vector where only the correct class index is marked with a 1, making them suitable for classification with softmax outputs.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Model
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score

# Load and preprocess the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, axis=-1)  # Add channel dimension
x_test = np.expand_dims(x_test, axis=-1)

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


This section defines and trains a convolutional neural network using Keras’ Functional API. The model takes images of shape (28, 28, 1) as input and applies two convolutional layers: the first with 32 filters and the second with 64, each using 3×3 kernels and ReLU activations to capture spatial features. Each convolutional layer is followed by max pooling to reduce dimensionality while retaining key information. The feature maps are then flattened and passed through a dense layer with 128 neurons activated by ReLU, before reaching the output layer with 10 softmax units, one for each digit class. The model is compiled with the Adam optimiser, categorical cross-entropy loss, and accuracy as the evaluation metric. Training is performed for three epochs with a batch size of 128, using 10% of the training data as a validation split to monitor generalisation performance.

In [ ]:
# Define the model using Functional API
def create_model():
    inputs = layers.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(10, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Train the model
print("Training the model...")
model = create_model()
model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1)

Training the model...
Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 44s 101ms/step - accuracy: 0.8542 - loss: 0.5255 - val_accuracy: 0.9802 - val_loss: 0.0670
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 43s 102ms/step - accuracy: 0.9788 - loss: 0.0673 - val_accuracy: 0.9865 - val_loss: 0.0442
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 81s 99ms/step - accuracy: 0.9874 - loss: 0.0407 - val_accuracy: 0.9865 - val_loss: 0.0451


This function generates adversarial examples using the Fast Gradient Sign Method (FGSM). The process begins by converting the input images and labels into TensorFlow tensors, enabling gradient-based operations. A GradientTape context records the forward pass of the model and computes the categorical cross-entropy loss between the true labels and the model’s predictions. The gradient of this loss with respect to the input images is then calculated, identifying how each pixel could be adjusted to increase error. By adding a small perturbation in the direction of the gradient sign, scaled by a factor ε, new inputs are created that are visually similar to the originals but can mislead the model. Finally, the perturbed images are clipped to remain within the valid pixel range [0, 1]. In this example, adversarial examples are generated for the first 100 test samples.

In [ ]:
# Generate adversarial examples using FGSM
def generate_adversarial_examples(model, x, y, epsilon=0.1):
    """
    Generates adversarial examples using FGSM.
    :param model: Trained model
    :param x: Input images
    :param y: True labels
    :param epsilon: Perturbation magnitude
    :return: Adversarial examples
    """
    x_tensor = tf.convert_to_tensor(x)
    y_tensor = tf.convert_to_tensor(y)

    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        predictions = model(x_tensor)
        loss = tf.keras.losses.categorical_crossentropy(y_tensor, predictions)

    gradients = tape.gradient(loss, x_tensor)
    adversarial_examples = x_tensor + epsilon * tf.sign(gradients)
    return tf.clip_by_value(adversarial_examples, 0.0, 1.0).numpy()

x_adv = generate_adversarial_examples(model, x_test[:100], y_test[:100])  # Generate for first 100 samples

This section introduces a simple statistical anomaly detection mechanism to identify adversarial examples. The method is based on monitoring the model’s prediction confidence: after running inputs through the trained model, the highest softmax probability for each sample is taken as its confidence score. If this maximum confidence falls below a defined threshold (here set at 0.8), the sample is flagged as anomalous, under the assumption that adversarial examples often produce less certain predictions than clean inputs. The function returns a Boolean array indicating which samples are detected as anomalies. To demonstrate this, the detection is applied to both clean test images and adversarially perturbed images, providing a comparison of how well the method distinguishes between the two.

In [ ]:
# Detection Mechanism 1 - Statistical Anomaly Detection
def detect_statistical_anomalies(model, x, threshold=0.8):
    """
    Detects adversarial examples based on confidence score anomalies.
    :param model: Trained model
    :param x: Input images
    :param threshold: Confidence threshold for detection
    :return: List of detected anomalies (True if detected)
    """
    predictions = model.predict(x)
    max_confidences = np.max(predictions, axis=1)
    anomalies = max_confidences < threshold
    return anomalies

# Detect adversarial examples
anomalies_clean = detect_statistical_anomalies(model, x_test[:100])  # Clean samples
anomalies_adv = detect_statistical_anomalies(model, x_adv)  # Adversarial samples

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


This section implements a detection method known as neural fingerprinting, which leverages the internal activation patterns of a network to identify adversarial examples. A “fingerprint” is first created by passing a set of clean reference samples through the network and extracting activations from the second-to-last layer, which captures high-level feature representations. The mean activation across these samples is computed to form the reference fingerprint. For detection, new inputs are run through the same intermediate model to obtain their activations. The Euclidean distance between these activations and the reference fingerprint is then calculated, with samples exceeding a chosen threshold flagged as anomalies. The idea is that adversarial inputs will cause deviations in internal representations compared to clean data, even if the final prediction appears confident.

In [ ]:
# Detection Mechanism 2 - Neural Fingerprinting
def create_neural_fingerprint(model, x_samples):
    """
    Creates a neural fingerprint by extracting intermediate layer activations.
    :param model: Trained model
    :param x_samples: Input images to generate fingerprints
    :return: Neural fingerprints (mean activation patterns)
    """
    # Build and call the intermediate model explicitly
    intermediate_model = Model(
        inputs=model.input,
        outputs=model.get_layer(index=-2).output  # Extract the second-to-last layer's activations
    )
    # Pass the samples through the intermediate model to get activations
    activations = intermediate_model.predict(x_samples)

    # Calculate the mean activation pattern as the fingerprint
    fingerprint = np.mean(activations, axis=0)
    return fingerprint

def detect_fingerprint_anomalies(model, x, reference_fingerprint, threshold=1.0):
    """
    Detects adversarial examples based on deviation from a reference fingerprint.
    :param model: Trained model
    :param x: Input images
    :param reference_fingerprint: Reference fingerprint for comparison
    :param threshold: Anomaly threshold
    :return: List of detected anomalies (True if detected)
    """
    intermediate_model = Model(
        inputs=model.input,
        outputs=model.get_layer(index=-2).output  # Use the second-to-last layer
    )
    activations = intermediate_model.predict(x)
    distances = np.linalg.norm(activations - reference_fingerprint, axis=1)
    anomalies = distances > threshold
    return anomalies

In this final step, the model’s detection methods are put into practice and compared. First, a reference fingerprint is generated from 100 clean test samples, representing the baseline activation pattern for neural fingerprinting. Both clean and adversarial samples are then evaluated against this reference to identify anomalies. The results are printed side by side: for statistical anomaly detection, the number of clean and adversarial examples flagged based on low confidence scores is shown, while for neural fingerprinting, the counts reflect deviations in internal activations relative to the reference fingerprint. This comparison highlights how different detection strategies vary in sensitivity and accuracy when distinguishing clean inputs from adversarial ones.

In [ ]:
# Generate reference fingerprint from clean samples
reference_fingerprint = create_neural_fingerprint(model, x_test[:100])

# Detect anomalies in clean and adversarial samples
fingerprint_anomalies_clean = detect_fingerprint_anomalies(model, x_test[:100], reference_fingerprint)
fingerprint_anomalies_adv = detect_fingerprint_anomalies(model, x_adv, reference_fingerprint)

# Step 6: Compare detection rates
print("Statistical Anomaly Detection:")
print(f"Clean samples detected as anomalies: {np.sum(anomalies_clean)} / 100")
print(f"Adversarial samples detected as anomalies: {np.sum(anomalies_adv)} / 100")

print("\nNeural Fingerprinting:")
print(f"Clean samples detected as anomalies: {np.sum(fingerprint_anomalies_clean)} / 100")
print(f"Adversarial samples detected as anomalies: {np.sum(fingerprint_anomalies_adv)} / 100")


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Statistical Anomaly Detection:
Clean samples detected as anomalies: 1 / 100
Adversarial samples detected as anomalies: 16 / 100

Neural Fingerprinting:
Clean samples detected as anomalies: 100 / 100
Adversarial samples detected as anomalies: 100 / 100
